In [ ]:
import logging
import random
from datetime import datetime, timedelta
import great_expectations as gx
from faker import Faker
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit, when, col
from pyspark.sql.types import StructType, StructField, LongType, TimestampType, StringType

In [ ]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

In [ ]:
schema = StructType([
    StructField("id", LongType(), False),
    StructField("published", TimestampType(), True),
    StructField("subject", StringType(), True),
    StructField("keyword", StringType(), True),
    StructField("title", StringType(), True),
    StructField("summary", StringType(), True),
    StructField("description", StringType(), True),
    StructField("original_link", StringType(), True),
    StructField("link", StringType(), True),
    StructField("created_at", TimestampType(), True),
    StructField("updated_at", TimestampType(), True),
])

In [ ]:
def random_dt(within_days=30):
    return datetime.now() - timedelta(
        days=random.randint(0, within_days),
        hours=random.randint(0, 23),
        minutes=random.randint(0, 59),
        seconds=random.randint(0, 59),
    )


def generate_news_data(fake: Faker, n: int, start_id: int = 1):
    return [generate_news_row(fake, start_id + i) for i in range(n)]


def generate_news_row(fake: Faker, id_: int):
    published = random_dt(14)
    return {
        "id": id_,
        "published": published,
        "subject": random.choice(["경제", "사회", "정치", "IT", "국제", "문화"]),
        "keyword": ", ".join(fake.words(nb=random.randint(2, 6)))[:200],
        "title": fake.sentence(nb_words=12)[:1000],
        "summary": fake.text(max_nb_chars=600)[:4000],
        "description": fake.text(max_nb_chars=2500),
        "original_link": fake.url()[:500],
        "link": fake.url()[:500],
        "created_at": published,
        "updated_at": published,
    }

In [ ]:
spark = SparkSession.builder.appName("news-dummy") \
    .master("spark://spark-master.mmix.io:7077") \
    .config("spark.sql.shuffle.partitions", "1") \
    .getOrCreate()

In [ ]:
news = spark.createDataFrame(generate_news_data(Faker("ko_KR"), 10000), schema=schema)
news_bad = news \
    .withColumn("title", when(col("id") % 200 == 0, lit(None)).otherwise(col("title"))) \
    .withColumn("link", when(col("id") % 333 == 0, lit("not-a-url")).otherwise(col("link")))
#news_bad.select(col("id"), col("title")).show(1, truncate=False)

In [ ]:
subjects = ["경제", "사회", "정치", "IT", "국제", "문화"]
url_regex = r"^https?://.+"

In [ ]:
context = gx.get_context(mode="ephemeral")
ds = context.data_sources.add_spark(name="spark_local")
asset = ds.add_dataframe_asset(name="news")
batch_def = asset.add_batch_definition_whole_dataframe("whole_df")
batch_parameters = {"dataframe": news_bad}
batch = batch_def.get_batch(batch_parameters=batch_parameters)
exp1 = gx.expectations.ExpectColumnValuesToNotBeNull(column="id")
exp2 = gx.expectations.ExpectColumnValuesToBeUnique(column="id")
exp3 = gx.expectations.ExpectColumnValuesToBeInSet(column="subject", value_set=subjects)
exp4 = gx.expectations.ExpectColumnValueLengthsToBeBetween(column="title", min_value=1, max_value=1000, mostly=0.999)
exp5 = gx.expectations.ExpectColumnValuesToMatchRegex(column="link", regex=url_regex, mostly=0.999)
results = [batch.validate(exp1), batch.validate(exp2), batch.validate(exp3), batch.validate(exp4), batch.validate(exp5)]

In [ ]:
success_all = all(r.success for r in results)
logging.info("all success: %s", success_all)
for result in results:
    logging.info(f"{result.expectation_config.type} => {result.success}")